# 第11章：目标设定与迭代（Goal Setting & Iteration）

## 概念

**目标设定与迭代** = 设定目标 → 生成 → 评估是否达标 → 不达标则改进 → 重复

```
设定目标 → 生成初稿 → 评估(是否达标?) → 否 → 改进 → 评估...
                              ↓ 是
                           输出结果
```

## 与第4章反思的区别

| | 第4章 反思 | 第11章 目标迭代 |
|--|-----------|----------------|
| **重点** | 找出错误/不足 | 对照目标检查 |
| **标准** | 通用质量 | 明确的目标列表 |
| **终止** | 固定次数 | 目标全部达成 |

## 代码演示

设定写作目标，让 LLM 迭代改进直到所有目标达成

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
load_dotenv()

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0
)

print("MiMo 模型初始化完成！")

MiMo 模型初始化完成！


## Step 1：设定目标

为写作任务设定明确的目标列表

In [3]:
# 定义写作目标
goals = [
    "故事字数在100-200字之间",
    "包含一个意外结局",
    "语言简洁生动",
    "主题围绕'信任'"
]

print("=== 写作目标 ===")
for i, goal in enumerate(goals, 1):
    print(f"  {i}. {goal}")

=== 写作目标 ===
  1. 故事字数在100-200字之间
  2. 包含一个意外结局
  3. 语言简洁生动
  4. 主题围绕'信任'


## Step 2：生成初稿

根据目标生成初始故事

In [4]:
# 生成故事的提示词
generate_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个作家。请根据以下目标写一个故事：

目标：
{goals}

请直接输出故事内容。"""),
    ("user", "请写一个故事。")
])

# 创建链
generate_chain = generate_prompt | llm | StrOutputParser()

# 生成初稿
goals_text = "\n".join([f"- {g}" for g in goals])
draft = generate_chain.invoke({"goals": goals_text})

print("=== 初稿 ===")
print(draft)

=== 初稿 ===
老陈的古董店来了个年轻人，捧着个锦盒，说是祖传的玉佩，急用钱，只押三千。老陈细看那玉，温润无瑕，是真品。他信任年轻人眼里的诚恳，没多问便付了钱。

三日后，年轻人没来赎。老陈打开锦盒复看，玉佩竟裂了道细纹。他心一沉，却见纹路里嵌着极小的字：“赝品，勿信。”字迹是年轻人的。

老陈愣住，随即笑了。他早看出那是高仿，却因那眼神而犹豫。原来年轻人用自污的方式，替他守住了信任——也守住了老陈差点因心软而亏掉的店。


## Step 3：评估是否达标

检查故事是否满足所有目标

In [5]:
# 评估提示词
evaluate_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个评审员。请检查故事是否满足以下目标：

目标：
{goals}

故事：
{story}

对每个目标，输出：
- 达成 / 未达成：原因

最后总结：全部达成 / 未达成"""),
    ("user", "请评估故事。")
])

# 创建链
evaluate_chain = evaluate_prompt | llm | StrOutputParser()

# 评估初稿
evaluation = evaluate_chain.invoke({"goals": goals_text, "story": draft})

print("=== 评估结果 ===")
print(evaluation)

=== 评估结果 ===
### 目标评估：

- **故事字数在100-200字之间**  
  达成：故事文本约130字，符合字数要求。

- **包含一个意外结局**  
  达成：结局揭示年轻人并非骗子，而是通过自污方式保护老陈的信任，出乎读者意料。

- **语言简洁生动**  
  达成：语言直接精炼，如“温润无瑕”、“心一沉”等描述生动形象，无冗余。

- **主题围绕'信任'**  
  达成：故事核心围绕信任展开，从老陈的信任到年轻人的守护，主题明确。

### 总结：全部达成


## Step 4：迭代改进

如果未达成目标，根据评估结果改进故事，直到全部达成或达到最大次数

In [6]:
# 改进故事的提示词
improve_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个作家。请根据评估结果改进故事，确保满足所有目标。

目标：
{goals}

当前故事：
{current_story}

评估结果：
{evaluation}

请输出改进后的故事。"""),
    ("user", "请改进故事。")
])

# 创建链
improve_chain = improve_prompt | llm | StrOutputParser()

# 迭代改进函数
def iterate_until_goals(goals, max_iterations=3):
    goals_text = "\n".join([f"- {g}" for g in goals])
    
    # 生成初稿
    story = generate_chain.invoke({"goals": goals_text})
    print(f"\n=== 初稿 ===")
    print(story)
    
    for i in range(max_iterations):
        # 评估
        evaluation = evaluate_chain.invoke({"goals": goals_text, "story": story})
        print(f"\n=== 评估 (第{i+1}次) ===")
        print(evaluation)
        
        # 检查是否全部达成
        if "全部达成" in evaluation:
            print(f"\n✅ 所有目标在第{i+1}次评估时达成！")
            return story
        
        # 改进故事
        story = improve_chain.invoke({
            "goals": goals_text,
            "current_story": story,
            "evaluation": evaluation
        })
        print(f"\n=== 改进稿 (第{i+1}次) ===")
        print(story)
    
    print(f"\n⚠️ 达到最大迭代次数({max_iterations})，返回最新版本。")
    return story

# 运行迭代
print("\n" + "="*50)
print("开始目标迭代")
print("="*50)

final_story = iterate_until_goals(goals, max_iterations=3)

print("\n" + "="*50)
print("最终故事")
print("="*50)
print(final_story)


开始目标迭代



=== 初稿 ===
老陈又来借钱了。这是他第三次开口，前两次的五百和八百都像扔进深潭的石子，再没回音。

我看着他磨破的袖口，想起二十年前他背着我跑过三条街找诊所的夜晚。钱包里的两千块是女儿下周的补习费。

“最后一次。”我把钱推过去时，妻子在厨房摔了碗。

三个月后，老陈消失了。邻居说看见他半夜搬走，像阵潮湿的风。我苦笑，信任原来这么廉价。

直到昨天，女儿在旧书包夹层发现一张存单——整整两万，存款人是老陈。附页的字迹歪斜：“利息。当年你背我时，我发誓要让你女儿读最好的书。”

窗外开始下雨，我突然想起，二十年前那个夜晚，也下着这样的雨。



=== 评估 (第1次) ===
### 目标评估

- **故事字数在100-200字之间**：达成。故事总字数约为185字，符合要求。
- **包含一个意外结局**：达成。故事从老陈借钱消失的负面预期，转折为发现存单的感恩回报，结局出乎意料。
- **语言简洁生动**：达成。语言简洁直接，如“老陈又来借钱了”，并运用生动比喻（如“像扔进深潭的石子”）和情感描写，增强感染力。
- **主题围绕'信任'**：达成。故事围绕信任展开：主角基于过去信任借钱，经历信任危机，最终揭示信任的回报，主题明确。

### 总结
全部达成。

✅ 所有目标在第1次评估时达成！

最终故事
老陈又来借钱了。这是他第三次开口，前两次的五百和八百都像扔进深潭的石子，再没回音。

我看着他磨破的袖口，想起二十年前他背着我跑过三条街找诊所的夜晚。钱包里的两千块是女儿下周的补习费。

“最后一次。”我把钱推过去时，妻子在厨房摔了碗。

三个月后，老陈消失了。邻居说看见他半夜搬走，像阵潮湿的风。我苦笑，信任原来这么廉价。

直到昨天，女儿在旧书包夹层发现一张存单——整整两万，存款人是老陈。附页的字迹歪斜：“利息。当年你背我时，我发誓要让你女儿读最好的书。”

窗外开始下雨，我突然想起，二十年前那个夜晚，也下着这样的雨。


## 目标迭代流程

```
┌─────────────────────────────────────────────────────────┐
│                目标设定与迭代流程                         │
├─────────────────────────────────────────────────────────┤
│  设定目标（明确、可衡量）                                 │
│     ↓                                                   │
│  生成初稿                                                │
│     ↓                                                   │
│  评估（对照目标检查）                                     │
│     ↓                                                   │
│  全部达成？ ──是──→ 输出结果                              │
│     │                                                   │
│     否                                                  │
│     ↓                                                   │
│  改进故事                                                │
│     ↓                                                   │
│  回到评估                                                │
└─────────────────────────────────────────────────────────┘
```

## 目标设定原则

| 原则 | 说明 | 示例 |
|------|------|------|
| **明确** | 不模糊 | "100-200字" 而非 "不要太长" |
| **可衡量** | 能判断是否达成 | "包含意外结局" 而非 "有趣" |
| **有限** | 目标数量适中 | 3-5 个目标 |
| **不冲突** | 目标之间兼容 | 不要同时要求 "简短" 和 "详细" |

## 与其他模式的关系

| 第4章 反思 | 第6章 规划 | 第9章 适应 | 第11章 目标迭代 |
|-----------|----------|----------|---------------|
| 生成→批评→改进 | 计划→执行 | 根据反馈改进 | 目标→评估→改进 |
| 通用质量改进 | 结构化执行 | 环境/反馈驱动 | 目标驱动的迭代 |

## 实际应用场景

- **代码生成**：设定功能、性能、可读性目标，迭代改进
- **文案创作**：设定字数、风格、情感目标
- **方案设计**：设定成本、可行性、效果目标
- **翻译**：设定准确度、流畅度、专业术语目标